# 03 - Scan Almeria and label what the model finds

1. Download PNOA orthophotos (Spanish IGN, free, CC BY 4.0) for an area of Almeria, in 1 km chunks (cached on Drive).
2. Score every 210 m window with the classifier from notebook 01 (and draw boxes with the detector from 02).
3. Save candidate sites as GeoJSON (opens in QGIS / Google Earth / geojson.io) and an interactive map.
4. Make a **labelling sheet**: top-scoring windows, windows near the threshold, and a random sample.
   Label them below (or in Excel). These labels are what notebook 04 uses to adapt the model to Almeria
   and to measure it honestly there.

Credit for imagery: "PNOA cedido por (c) Instituto Geografico Nacional de Espana".

In [ ]:
# Setup: GPU, Google Drive (click "Connect to Google Drive" when asked), code
import os, sys, subprocess, json, pathlib
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
from google.colab import drive
drive.mount("/content/drive")
RUNS = pathlib.Path("/content/drive/MyDrive/waste_runs"); RUNS.mkdir(parents=True, exist_ok=True)
REPO = "/content/waste_detection"
if not os.path.exists(REPO):
    !git clone -q https://github.com/inayatarshad/waste_detection.git {REPO}
else:
    !git -C {REPO} pull -q
!pip install -q -r {REPO}/requirements.txt
sys.path.insert(0, f"{REPO}/src")
import numpy as np, torch
print("torch", torch.__version__, "| GPU:", torch.cuda.is_available())

In [ ]:
from wastedet import geo, scan
from wastedet.train_cls import load_classifier
AOIS = json.loads(pathlib.Path(f"{REPO}/configs/almeria_aoi.json").read_text())["areas"]
AOI_NAME = "nijar_demo"                 # start small; then "nijar_campo", "campo_de_dalias"
GSD = 0.3                                # metres per pixel requested from the WMS
aoi = AOIS[AOI_NAME]
chunks = geo.scan_chunks(aoi["bbox_lonlat"], window_m=210, stride_m=105, windows_per_side=8, gsd=GSD)
CHUNK_CACHE = RUNS / "pnoa_cache"
print(f"{AOI_NAME}: {aoi['note']}\n{len(chunks)} chunks of {chunks[0].size_m:.0f} m ({chunks[0].pixels} px)")
model, ckpt = load_classifier(RUNS / "classifier_v1" / "best.pt", "cuda")
TYPES, THR = ckpt["types"], ckpt["thresholds"]

In [ ]:
# Download + scan (re-running continues where it stopped; downloads are cached on Drive)
from tqdm.auto import tqdm
SCAN = RUNS / "almeria_scan" / AOI_NAME; SCAN.mkdir(parents=True, exist_ok=True)
windows = []
for c in tqdm(chunks):
    try:
        img = geo.fetch_wms(c, cache_dir=CHUNK_CACHE)
    except RuntimeError as e:
        print("skipped", c.key, e); continue
    if geo.is_empty(img):
        continue
    w = scan.scan_image(model, img, gsd=c.gsd, window_m=210, stride_m=105, size=640, batch=32,
                        device="cuda", tta=True, exact_grid=True)
    windows += scan.add_map_coords(w, c.x0, c.y1, c.gsd)
(SCAN / "windows.json").write_text(json.dumps(windows))
s = np.array([w["waste"] for w in windows])
print(f"{len(windows)} windows scanned; above threshold {THR['waste']:.2f}: {(s >= THR['waste']).sum()}")

In [ ]:
# Candidate sites -> GeoJSON + interactive map
sites = scan.cluster_windows(windows, THR["waste"], stride_m=105)
fc = scan.sites_to_geojson(sites, TYPES, SCAN / "candidate_sites.geojson", THR["types"])
print(len(sites), "candidate sites (highest first):")
for f in fc["features"][:10]:
    p = f["properties"]; print(p["site"], p["waste_score"], "plastic", p.get("Plastic_score"), p["lat"], p["lon"])
import folium
lon0, lat0, lon1, lat1 = aoi["bbox_lonlat"]
m = folium.Map(location=[(lat0 + lat1) / 2, (lon0 + lon1) / 2], zoom_start=13, tiles=None)
folium.raster_layers.WmsTileLayer(url=geo.PNOA_WMS, layers=geo.PNOA_LAYER, name="PNOA", fmt="image/jpeg",
                                  attr=geo.PNOA_CREDIT).add_to(m)
folium.GeoJson(fc, name="candidate sites", tooltip=folium.GeoJsonTooltip(["site", "waste_score", "Plastic_score"]),
               style_function=lambda f: {"color": "red" if f["properties"].get("Plastic_flag") else "orange", "weight": 2}).add_to(m)
m.save(str(SCAN / "map.html")); m

In [ ]:
# Close-ups of the top sites, with detector boxes when the detector from notebook 02 exists
from wastedet import plots
from wastedet.dumpsites import DumpsiteDetDataset
from wastedet.imageio_utils import resize_square
det = None
if (RUNS / "detector_v1" / "best_det.pt").exists():
    from wastedet.train_det import load_detector
    det, dck = load_detector(RUNS / "detector_v1" / "best_det.pt", "cuda")
FIG = SCAN / "figures"; FIG.mkdir(exist_ok=True)
for s_ in sites[:8]:
    crop = scan.crop_from_chunks(chunks, CHUNK_CACHE, s_["x"], s_["y"], 300, GSD)
    if crop is None:
        continue
    img = resize_square(crop, 800)
    boxes, scores = np.zeros((0, 4)), np.zeros(0)
    if det is not None:
        with torch.no_grad():
            o = det([torch.from_numpy(img.transpose(2, 0, 1) / 255.0).float().cuda()])[0]
        boxes, scores = o["boxes"].cpu().numpy(), o["scores"].cpu().numpy()
    lon, lat = geo.utm_to_lonlat(s_["x"], s_["y"])
    plots.draw_boxes(img, [], boxes, scores, dck["score_threshold"] if det else 1.0, FIG / f"site_{lat:.5f}_{lon:.5f}.png",
                     title=f"waste {s_['score']:.2f} plastic {s_['types_max'][0]:.2f}")
from IPython.display import Image as IImage, display
for f in sorted(FIG.glob("site_*.png"))[:8]:
    display(IImage(str(f)))

In [ ]:
# Labelling sheet: 300 top + 100 near-threshold + 300 random windows (random = honest false-alarm/miss rates)
LAB = RUNS / "almeria_labels"; LAB.mkdir(exist_ok=True)
SHEET = LAB / f"sheet_{AOI_NAME}.csv"
if not SHEET.exists():
    samples = scan.sample_for_labelling(windows, n_top=300, n_random=300, n_border=100, thr=THR["waste"])
    def get_crop(s_):
        return scan.crop_from_chunks(chunks, CHUNK_CACHE, s_["x"], s_["y"], 210, GSD)
    scan.write_sheet(samples, LAB / "crops", SHEET, get_crop)
print("sheet:", SHEET)

### Label the windows
For each 210 m image: **is there dumped waste?** and if yes, **is plastic part of it?** Also tick whether greenhouses
are visible (lets us measure false alarms on greenhouses). Intact greenhouses, plastic mulch on fields,
and plastic stored neatly at a recycling yard are **not** waste. Model scores are hidden on purpose so they do not
influence you. Progress is saved to Drive after every click; you can stop and continue later.

In [ ]:
import pandas as pd, ipywidgets as W
from IPython.display import display, clear_output
sheet = pd.read_csv(SHEET, dtype=str).fillna("")
todo = [i for i in range(len(sheet)) if sheet.at[i, "waste"] == ""]
state = {"k": 0}
img_w = W.Image(width=520); info = W.HTML(); gh = W.Checkbox(description="greenhouses visible")
def show():
    if state["k"] >= len(todo):
        info.value = "<b>All done.</b>"; return
    i = todo[state["k"]]
    img_w.value = (LAB / "crops" / sheet.at[i, "file"]).read_bytes()
    info.value = f"{state['k'] + 1} / {len(todo)} &nbsp; ({sheet.at[i, 'lat']}, {sheet.at[i, 'lon']})"
    gh.value = False
def answer(waste, plastic):
    def f(_):
        if state["k"] < len(todo):
            i = todo[state["k"]]
            if waste is not None:
                sheet.at[i, "waste"], sheet.at[i, "plastic"] = waste, plastic
                sheet.at[i, "greenhouse"] = "1" if gh.value else "0"
            sheet.to_csv(SHEET, index=False); state["k"] += 1; show()
    return f
buttons = [("Waste + plastic", "1", "1"), ("Waste, no plastic", "1", "0"), ("Waste, plastic unsure", "1", ""),
           ("No waste", "0", "0"), ("Skip", None, None)]
row = W.HBox([W.Button(description=d, layout=W.Layout(width="170px")) for d, *_ in buttons])
for b, (_, w, p) in zip(row.children, buttons):
    b.on_click(answer(w, p))
display(W.VBox([info, img_w, gh, row])); show()